In [1]:
!nvidia-smi

Mon Apr 27 21:19:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB


In [3]:
import os, getpass
token = getpass.getpass("GitHub token: ")
!git clone https://{token}@github.com/jasmineztruong8/efficient-codegen.git
%cd /content/efficient-codegen

!git fetch origin
!git checkout jg/serving-benchmarks
!git pull origin jg/serving-benchmarks
!git log --oneline -n 10


GitHub token: ··········
Cloning into 'efficient-codegen'...
remote: Enumerating objects: 285, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 285 (delta 72), reused 76 (delta 25), pack-reused 140 (from 1)
Receiving objects: 100% (285/285), 25.95 MiB | 11.36 MiB/s, done.
Resolving deltas: 100% (130/130), done.
/content/efficient-codegen
Branch 'jg/serving-benchmarks' set up to track remote branch 'jg/serving-benchmarks' from 'origin'.
Switched to a new branch 'jg/serving-benchmarks'
From https://github.com/jasmineztruong8/efficient-codegen
 * branch            jg/serving-benchmarks -> FETCH_HEAD
Already up to date.
f489fb7 (HEAD -> jg/serving-benchmarks, origin/jg/serving-benchmarks) add notebook and results
c318d35 comparison runs
1ba36ea Merge pull request #5 from jasmineztruong8/amahajan-train-run
b4ec13a (origin/amahajan-train-run) add requirements
2d20775 Merge pull request #9 from jasmineztruong8/model-skeleto

In [4]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl torch pandas
!pip install -q vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 46.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jt3595 (efficient-codegen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [19]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
RUNTIME_AWARE_ADAPTER = "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full"
RUNTIME_AWARE_MERGED = "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged"

INPUT_PATH = "data/curated/scale1k/dataset_clean.json"
SERVING_OUTPUT_DIR = "outputs/serving"

WANDB_PROJECT = "hpml-efficient-codegen"

In [ ]:
!ls serving
!ls training
!ls data/curated/scale1k

benchmark_serving.py  merge_checkpoint.py
data  evaluate_model.py  select_training_data.py  train.py
benchmark_results.json	dataset_clean.json


In [ ]:
!python serving/merge_checkpoint.py \
  --adapter_path {RUNTIME_AWARE_ADAPTER} \
  --base_model_name {BASE_MODEL} \
  --output_dir {RUNTIME_AWARE_MERGED}

Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
config.json: 100% 660/660 [00:00<00:00, 3.32MB/s]
tokenizer_config.json: 7.30kB [00:00, 12.9MB/s]
vocab.json: 2.78MB [00:00, 112MB/s]
merges.txt: 1.67MB [00:00, 116MB/s]
tokenizer.json: 7.03MB [00:00, 37.8MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:08<00:00, 366MB/s]
Loading weights: 100% 338/338 [00:00<00:00, 421.01it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.28MB/s]
Loading adapter: /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full
Merging adapter weights into base model...
Saving merged model to /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
Writing model shards: 100% 1/1 [00:09<00:00,  9.84s/it]
Done.


In [ ]:
# smoke: base + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/base_hf_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_hf_smoke


Loaded 100 prompts
Running Hugging Face benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 438.51it/s]
{
  "backend": "hf",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 100,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.8423035143199991,
  "avg_batch_latency_s": 6.475257196153839,
  "throughput_prompts_per_s": 1.1872205006853271,
  "throughput_output_tokens_per_s": 217.02390752527782,
  "peak_cuda_memory_mb": 3507.087890625,
  "avg_gpu_util_pct": 36.79503105590062,
  "max_gpu_util_pct": 55.0,
  "avg_gpu_mem_mb": 4134.6459627329195,
  "max_gpu_mem_mb": 4150.0
}
Results saved to outputs/serving/base_hf_smoke.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --r

In [ ]:
# smoke: base + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/base_vllm_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_vllm_smoke


Loaded 100 prompts
Running vLLM benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
INFO 04-20 19:49:33 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct'}
INFO 04-20 19:49:50 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 19:49:50 [model.py:1678] Using max model len 32768
INFO 04-20 19:49:50 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 19:49:50 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 19:49:52 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=10228) INFO 04-20 19:50:06 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen2.5-Coder-1.5B-Instruct', sp

In [ ]:
# smoke: runtime aware + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_hf_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_hf_smoke

Loaded 100 prompts
Running Hugging Face benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 175.34it/s]
{
  "backend": "hf",
  "model_path": "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged",
  "num_prompts": 100,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.7775845633000017,
  "avg_batch_latency_s": 5.977211773307632,
  "throughput_prompts_per_s": 1.2860337604389758,
  "throughput_output_tokens_per_s": 220.16897978715267,
  "peak_cuda_memory_mb": 3507.087890625,
  "avg_gpu_util_pct": 36.95302013422819,
  "max_gpu_util_pct": 45.0,
  "avg_gpu_mem_mb": 4139.463087248322,
  "max_gpu_mem_mb": 4150.0
}
Results saved to outputs/serving/runtime_aware_hf_smoke.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logge

In [ ]:
# smoke: runtime aware + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_vllm_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_vllm_smoke

Loaded 100 prompts
Running vLLM benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
INFO 04-20 19:53:10 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged'}
INFO 04-20 19:53:10 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 19:53:10 [model.py:1678] Using max model len 32768
INFO 04-20 19:53:10 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 19:53:10 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 19:53:13 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=11822) INFO 04-20 19:53:26 [core.py:105] Initializing a

In [ ]:
!ls {SERVING_OUTPUT_DIR}
!cat {SERVING_OUTPUT_DIR}/base_hf_smoke.json
!cat {SERVING_OUTPUT_DIR}/base_vllm_smoke.json
!cat {SERVING_OUTPUT_DIR}/runtime_aware_hf_smoke.json
!cat {SERVING_OUTPUT_DIR}/runtime_aware_vllm_smoke.json

base_hf_smoke.json    runtime_aware_hf_smoke.json
base_vllm_smoke.json  runtime_aware_vllm_smoke.json
{
  "backend": "hf",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 100,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.8423035143199991,
  "avg_batch_latency_s": 6.475257196153839,
  "throughput_prompts_per_s": 1.1872205006853271,
  "throughput_output_tokens_per_s": 217.02390752527782,
  "peak_cuda_memory_mb": 3507.087890625,
  "avg_gpu_util_pct": 36.79503105590062,
  "max_gpu_util_pct": 55.0,
  "avg_gpu_mem_mb": 4134.6459627329195,
  "max_gpu_mem_mb": 4150.0
}{
  "backend": "vllm",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 100,
  "batch_size": 32,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.1180070548599997,
  "avg_batch_latency_s": 2.9501583009999877,
  "throughput_prompts_per_s": 8.474069632416233,
  "throughput_out

In [ ]:
# full: base + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/base_hf_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_hf_full


Loaded 1000 prompts
Running Hugging Face benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 435.12it/s]
{
  "backend": "hf",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 1000,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.6887423910689999,
  "avg_batch_latency_s": 5.506712247592008,
  "throughput_prompts_per_s": 1.451921666165917,
  "throughput_output_tokens_per_s": 233.51546541279677,
  "peak_cuda_memory_mb": 3705.791015625,
  "avg_gpu_util_pct": 37.62290076335878,
  "max_gpu_util_pct": 73.0,
  "avg_gpu_mem_mb": 4455.21679389313,
  "max_gpu_mem_mb": 4902.0
}
Results saved to outputs/serving/base_hf_full.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --rel

In [ ]:
# full: base + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/base_vllm_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_vllm_full


Loaded 1000 prompts
Running vLLM benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
INFO 04-20 20:07:38 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct'}
INFO 04-20 20:07:40 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 20:07:40 [model.py:1678] Using max model len 32768
INFO 04-20 20:07:40 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 20:07:40 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 20:07:43 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=18653) INFO 04-20 20:07:57 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen2.5-Coder-1.5B-Instruct', s

In [ ]:
# full: runtime aware + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_hf_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_hf_full


Loaded 1000 prompts
Running Hugging Face benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 189.04it/s]
{
  "backend": "hf",
  "model_path": "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged",
  "num_prompts": 1000,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.6695119966330003,
  "avg_batch_latency_s": 5.352918561072005,
  "throughput_prompts_per_s": 1.4936252151253984,
  "throughput_output_tokens_per_s": 228.62024992795395,
  "peak_cuda_memory_mb": 3705.791015625,
  "avg_gpu_util_pct": 37.285490196078435,
  "max_gpu_util_pct": 92.0,
  "avg_gpu_mem_mb": 4475.345882352941,
  "max_gpu_mem_mb": 4902.0
}
Results saved to outputs/serving/runtime_aware_hf_full.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently log

In [ ]:
# full runtime aware + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_vllm_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_vllm_full


Loaded 1000 prompts
Running vLLM benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
INFO 04-20 20:20:48 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged'}
INFO 04-20 20:20:48 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 20:20:48 [model.py:1678] Using max model len 32768
INFO 04-20 20:20:48 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 20:20:48 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 20:20:50 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=25050) INFO 04-20 20:21:04 [core.py:105] Initializing 

In [ ]:
import json
import pandas as pd

paths = [
    "outputs/serving/base_hf_full.json",
    "outputs/serving/base_vllm_full.json",
    "outputs/serving/runtime_aware_hf_full.json",
    "outputs/serving/runtime_aware_vllm_full.json",
]

rows = []
for path in paths:
    with open(path, "r") as f:
        rows.append(json.load(f))

df = pd.DataFrame(rows)
df.to_csv("outputs/serving/serving_full_results.csv", index=False)
df


,backend,model_path,num_prompts,batch_size,max_new_tokens,temperature,top_p,avg_latency_per_prompt_s,avg_batch_latency_s,throughput_prompts_per_s,throughput_output_tokens_per_s,peak_cuda_memory_mb,avg_gpu_util_pct,max_gpu_util_pct,avg_gpu_mem_mb,max_gpu_mem_mb
0,hf,Qwen/Qwen2.5-Coder-1.5B-Instruct,1000,8,256,0.2,0.95,0.688742,5.506712,1.451922,233.515465,3705.791016,37.622901,73.0,4455.216794,4902.0
1,vllm,Qwen/Qwen2.5-Coder-1.5B-Instruct,1000,32,256,0.2,0.95,0.031796,0.993604,31.450586,1781.612794,37602.000000,97.213115,100.0,37500.983607,37602.0
2,hf,/content/drive/MyDrive/efficient-codegen/check...,1000,8,256,0.2,0.95,0.669512,5.352919,1.493625,228.620250,3705.791016,37.285490,92.0,4475.345882,4902.0
3,vllm,/content/drive/MyDrive/efficient-codegen/check...,1000,32,256,0.2,0.95,0.031674,0.989795,31.571594,1891.106930,37614.000000,96.770492,100.0,37510.885246,37614.0


In [ ]:
from google.colab import files
files.download("outputs/serving/serving_full_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Control SFT Serving Benchmark

In [20]:
CONTROL_ADAPTER = "/content/drive/MyDrive/HPML Final Project/t4_checkpoints/checkpoints/control"
CONTROL_MERGED = "/content/drive/MyDrive/HPML Final Project/t4_checkpoints/checkpoints/control_merged"
SERVING_OUTPUT_DIR = "/content/drive/MyDrive/HPML Final Project/outputs/serving"

In [23]:
!pip install -q torchao --upgrade
!pip install -q bitsandbytes --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.2 MB/s eta 0:00:00


In [24]:
!python serving/merge_checkpoint.py \
--adapter_path "$CONTROL_ADAPTER" \
--base_model_name {BASE_MODEL} \
--output_dir "$CONTROL_MERGED"

Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 422.34it/s]
Loading adapter: /content/drive/MyDrive/HPML Final Project/t4_checkpoints/checkpoints/control
bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._han

In [28]:
!python serving/benchmark_serving.py \
--backend hf \
--model_path "{CONTROL_MERGED}" \
--input_path {INPUT_PATH} \
--limit 1000 \
--batch_size 8 \
--output_path {SERVING_OUTPUT_DIR}/control_hf_full.json \
--use_wandb \
--wandb_project hpml-efficient-codegen \
--run_name control_hf_full

Loaded 1000 prompts
Running Hugging Face benchmark for /content/drive/MyDrive/HPML Final Project/t4_checkpoints/checkpoints/control_merged
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 173.11it/s]
{
  "backend": "hf",
  "model_path": "/content/drive/MyDrive/HPML Final Project/t4_checkpoints/checkpoints/control_merged",
  "num_prompts": 1000,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.6962052960139997,
  "avg_batch_latency_s": 5.566321929751986,
  "throughput_prompts_per_s": 1.4363579331058283,
  "throughput_output_tokens_per_s": 221.86559177925866,
  "peak_cuda_memory_mb": 3705.791015625,
  "avg_gpu_util_pct": 36.7286470143613,
  "max_gpu_util_pct": 78.0,
  "avg_gpu_mem_mb": 4482.527588813303,
  "max_gpu_mem_mb": 4902.0
}
Results saved to outputs/serving/control_hf_full.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.

In [40]:
!python serving/benchmark_serving.py \
--backend vllm \
--model_path "$CONTROL_MERGED" \
--input_path {INPUT_PATH} \
--limit 1000 \
--batch_size 32 \
--output_path "$SERVING_OUTPUT_DIR/control_vllm_full.json" \
--use_wandb \
--wandb_project hpml-efficient-codegen \
--run_name control_vllm_full

Loaded 1000 prompts
Running vLLM benchmark for /content/drive/MyDrive/HPML Final Project/t4_checkpoints/checkpoints/control_merged
INFO 04-27 22:55:09 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': '/content/drive/MyDrive/HPML Final Project/t4_checkpoints/checkpoints/control_merged'}
INFO 04-27 22:55:09 [model.py:555] Resolved architecture: Qwen2ForCausalLM
INFO 04-27 22:55:09 [model.py:1680] Using max model len 32768
INFO 04-27 22:55:09 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-27 22:55:09 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 04-27 22:55:09 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
WARNING 04-27 22:55:11 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/

In [41]:
import json
import pandas as pd

serving_dir = "/content/drive/MyDrive/HPML Final Project/outputs/serving"

paths = [
    f"{serving_dir}/control_hf_full.json",
    f"{serving_dir}/control_vllm_full.json",
]

rows = []
for path in paths:
    with open(path) as f:
        rows.append(json.load(f))

df_new = pd.DataFrame(rows)
df_existing = pd.read_csv("/content/efficient-codegen/outputs/serving_full_results.csv")
df_combined = pd.concat([df_existing, df_new], ignore_index=True)
df_combined.to_csv("/content/drive/MyDrive/HPML Final Project/outputs/serving_full_results.csv", index=False)
print(df_combined[["backend", "model_path", "throughput_prompts_per_s", "avg_latency_per_prompt_s", "avg_gpu_util_pct"]])

  backend                                         model_path  \
0      hf                   Qwen/Qwen2.5-Coder-1.5B-Instruct   
1    vllm                   Qwen/Qwen2.5-Coder-1.5B-Instruct   
2      hf  /content/drive/MyDrive/efficient-codegen/check...   
3    vllm  /content/drive/MyDrive/efficient-codegen/check...   
4      hf  /content/drive/MyDrive/HPML Final Project/t4_c...   
5    vllm  /content/drive/MyDrive/HPML Final Project/t4_c...   

   throughput_prompts_per_s  avg_latency_per_prompt_s  avg_gpu_util_pct  
0                  1.451922                  0.688742         37.622901  
1                 31.450586                  0.031796         97.213115  
2                  1.493625                  0.669512         37.285490  
3                 31.571594                  0.031674         96.770492  
4                  1.436358                  0.696205         36.728647  
5                 25.364144                  0.039426         78.040000  


In [42]:
from google.colab import files
files.download("/content/drive/MyDrive/HPML Final Project/outputs/serving_full_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>